# Train And Save SAE From A Trained Transformer

This notebook only does path setup, SAE training, and saving the SAE artifact for later analysis.

In [1]:
import os
import subprocess
from pathlib import Path

import torch

print(f'Current notebook cwd: {Path.cwd()}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device count: {torch.cuda.device_count()}')
    print(f'Current CUDA device: {torch.cuda.get_device_name(0)}')
else:
    print('Warning: CUDA not available, will run on CPU unless you set device manually.')

!nvidia-smi

Current notebook cwd: /home/ponsin
CUDA available: True
CUDA device count: 1
Current CUDA device: NVIDIA H100
Wed Mar 18 17:05:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100                    On  |   00000000:E4:00.0 Off |                    0 |
| N/A   36C    P0             65W /  700W |       5MiB /  95830MiB |      0%      Default |
|                             

## Configure Paths, SAE Hyperparameters, And SAE Data Splits

Use the consolidated transformer output file produced by `main.py` (typically ending with `.pt`, and in your Slurm naming often `.pkl.pt`).
Then choose where to save the trained SAE artifact, plus the train/eval RHM sizes used for SAE training vs later analysis.

In [ ]:
# Required input artifact from transformer training (single consolidated file from main.py)
train_output = Path('/work/pcsl/ponsin/Mean_Transformer/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt')

# Path to the exact train_sae.py script you want to run
train_sae_script = Path('/home/ponsin/SAE-on-RHM/train_sae.py')



# SAE training settings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sae_layers = 'all'        # e.g. 'all' or '0,1,2'
sae_latent_dim = 20*512     # None means 4 * embedding_dim
sae_lambda_l1 = 3
sae_lr = 5e-5
sae_steps = 2**12
sae_sample_batch_size = 2**10    # number of RHM samples per transformer forward
sae_batch_limit = 0           # 0 means no cap on activations used per SAE step
sae_print_freq = 2**8

# RHM sizes/seeds for SAE pipeline
sae_train_size = 2**17          # data used to train SAE
sae_eval_size = 2**17             # separate data reserved for later SAE analysis
sae_train_seed_sample = None      # None => transformer seed + 1 (different data, same rules)
sae_eval_seed_sample = None       # None => SAE train seed + 1

# Required output artifacts for the SAEs
sae_output_path = Path(f'/work/pcsl/ponsin/Mean_Transformer/SAE/v_16_L_3_m_4/SAE__v_16_L_3_m=4_P_{sae_train_size}_l1_{sae_lambda_l1}_ldim_{sae_latent_dim}_lr_{sae_lr}_steps_{sae_steps}_bsize_{sae_sample_batch_size}.pt')

print(f'train_output: {train_output}')
print(f'train_sae_script: {train_sae_script}')
print(f'sae_output_path: {sae_output_path}')
print(f'sae_sample_batch_size: {sae_sample_batch_size}')
print(f'sae_batch_limit: {sae_batch_limit}')
print(f'sae_train_size: {sae_train_size}')
print(f'sae_eval_size: {sae_eval_size}')
print(f'sae_train_seed_sample: {sae_train_seed_sample}')
print(f'sae_eval_seed_sample: {sae_eval_seed_sample}')

train_output: /work/pcsl/ponsin/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt
train_sae_script: /home/ponsin/SAE-on-RHM/train_sae.py
sae_output_path_all_tokens: /work/pcsl/ponsin/SAE/v_16_L_3_m_4/SAE_all_tokens_v_16_L_3_m=4_P_131072_l1_3_ldim_10240_lr_5e-05_steps_4096_bsize_1024.pt
sae_output_path_cls_token: /work/pcsl/ponsin/SAE/v_16_L_3_m_4/SAE_cls_token_v_16_L_3_m=4_P_131072_l1_3_ldim_10240_lr_5e-05_steps_4096_bsize_1024.pt
sae_sample_batch_size: 1024
sae_batch_limit: 0
sae_train_size: 131072
sae_eval_size: 131072
sae_train_seed_sample: None
sae_eval_seed_sample: None


In [ ]:
def _assert_exists(path_obj, name):
    if path_obj is None:
        raise ValueError(f'{name} is None')
    if not Path(path_obj).exists():
        raise FileNotFoundError(f'{name} not found: {path_obj}')

_assert_exists(train_output, 'train_output')
_assert_exists(train_sae_script, 'train_sae_script')

# Ensure this artifact carries the exact RHM rules used in transformer training.
blob = torch.load(train_output, map_location='cpu')
if not isinstance(blob, dict) or 'output' not in blob:
    raise ValueError(
        'train_output must be a consolidated main.py artifact containing an output dict.'
    )
if 'rules' not in blob['output']:
    raise ValueError(
        'train_output is missing output.rules. Re-run transformer training with the updated Sbatch_trsf_for_SAE.sh.'
    )
print('Verified train_output contains output.rules for fixed-RHM SAE training.')

def build_sae_cmd(sae_output_path, activation_source):
    cmd = [
        'python', str(train_sae_script),
        '--train_output', str(train_output),
        '--outname', str(sae_output_path),
        '--device', device,
        '--sae_layers', str(sae_layers),
        '--sae_lambda_l1', str(float(sae_lambda_l1)),
        '--sae_lr', str(float(sae_lr)),
        '--sae_steps', str(int(sae_steps)),
        '--sae_sample_batch_size', str(int(sae_sample_batch_size)),
        '--sae_batch_limit', str(int(sae_batch_limit)),
        '--sae_activation_source', str(activation_source),
        '--sae_print_freq', str(int(sae_print_freq)),
        '--sae_train_size', str(int(sae_train_size)),
        '--sae_eval_size', str(int(sae_eval_size)),
    ]

    if sae_latent_dim is not None:
        cmd += ['--sae_latent_dim', str(int(sae_latent_dim))]
    if sae_train_seed_sample is not None:
        cmd += ['--sae_train_seed_sample', str(int(sae_train_seed_sample))]
    if sae_eval_seed_sample is not None:
        cmd += ['--sae_eval_seed_sample', str(int(sae_eval_seed_sample))]

    return cmd

cmd_all_tokens = build_sae_cmd(sae_output_path, 'all_tokens')

print('Command (all_tokens):')
print(' '.join(cmd_all_tokens))

/tmp/2663192/ipykernel_3620858/356620418.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = torch.load(train_output, map_location='cpu')


Verified train_output contains output.rules for fixed-RHM SAE training.
Command (all_tokens):
python /home/ponsin/SAE-on-RHM/train_sae.py --train_output /work/pcsl/ponsin/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt --outname /work/pcsl/ponsin/SAE/v_16_L_3_m_4/SAE_all_tokens_v_16_L_3_m=4_P_131072_l1_3_ldim_10240_lr_5e-05_steps_4096_bsize_1024.pt --device cuda --sae_layers all --sae_lambda_l1 3.0 --sae_lr 5e-05 --sae_steps 4096 --sae_sample_batch_size 1024 --sae_batch_limit 0 --sae_activation_source all_tokens --sae_print_freq 256 --sae_train_size 131072 --sae_eval_size 131072 --sae_latent_dim 10240

Command (cls_token):
python /home/ponsin/SAE-on-RHM/train_sae.py --train_output /work/pcsl/ponsin/Transformer_for_SAE/v_16_L_3_m_4/RESULT_TRFCLASS_v_16_L_3_m=4_P_12160_0_emb_512_h_8_lr_5e-3_dropout_0.1.pkl.pt --outname /work/pcsl/ponsin/SAE/v_16_L_3_m_4/SAE_cls_token_v_16_L_3_m=4_P_131072_l1_3_ldim_10240_lr_5e-05_steps_4096_b

In [ ]:
def run_and_check(cmd, label):
    print(f'Running SAE training: {label}')
    result = subprocess.run(
        cmd,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'{label} failed with return code {result.returncode}')
    print(f'{label} finished successfully.')
    return result.stdout

sae_train_logs = {}
sae_train_logs['all_tokens'] = run_and_check(cmd_all_tokens, 'all_tokens SAE')

print('SAE trainings finished successfully.')

Running SAE training: all_tokens SAE
SAE data split: train_size=131072 (seed_sample=826178), eval_size=131072 (seed_sample=826179)
Transformer training seed_sample=826177
Using fixed RHM rules loaded from training artifact.
# parameters: 9465872
sae layer 0 step 0/4096 total=10617.845703 recon=3182.761475 sparse=2478.361328 active_fraction=0.500493 dead_features=0
sae layer 0 step 256/4096 total=3923.228516 recon=3923.211914 sparse=0.005556
sae layer 0 step 512/4096 total=3915.845215 recon=3915.841797 sparse=0.001141
sae layer 0 step 768/4096 total=3939.583252 recon=3939.582031 sparse=0.000432
sae layer 0 step 1024/4096 total=3924.716553 recon=3924.715088 sparse=0.000462
sae layer 0 step 1280/4096 total=3880.253662 recon=3880.252930 sparse=0.000224
sae layer 0 step 1536/4096 total=3898.123047 recon=3898.122070 sparse=0.000346
sae layer 0 step 1792/4096 total=3886.214111 recon=3886.213867 sparse=0.000061
sae layer 0 step 2048/4096 total=3954.591553 recon=3954.591309 sparse=0.000093
sae 

In [ ]:
for label, path in [('all_tokens SAE', sae_output_path), ('cls_token SAE', sae_output_path)]:
    p = Path(path)
    print(f'{label} output path: {p.resolve()}')
    if not p.exists():
        raise FileNotFoundError(f'Could not find {label} output at {p}')

print('SAE artifacts are ready for loading in a later step.')

all_tokens SAE output path: /work/pcsl/ponsin/SAE/v_16_L_3_m_4/SAE_all_tokens_v_16_L_3_m=4_P_131072_l1_3_ldim_10240_lr_5e-05_steps_4096_bsize_1024.pt
cls_token SAE output path: /work/pcsl/ponsin/SAE/v_16_L_3_m_4/SAE_cls_token_v_16_L_3_m=4_P_131072_l1_3_ldim_10240_lr_5e-05_steps_4096_bsize_1024.pt
Both SAE artifacts are ready for loading in a later step.


In [16]:
# Compact SAE analysis on a fresh evaluation split
import copy
import sys
from pathlib import Path

import torch

repo_root = Path(train_sae_script).resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import init
import models
from datasets.random_hierarchy_model import sample_trees

# -----------------------------
# Analysis configuration
# -----------------------------
analysis_eval_size = 2**15
analysis_eval_seed_sample = 98765432
analysis_batch_size = 256
analysis_device = device

In [7]:
def load_transformer_and_loader(train_output_path, eval_size, eval_seed, batch_size, device_name):
    blob = torch.load(train_output_path, map_location='cpu')
    if not isinstance(blob, dict) or 'config' not in blob or 'output' not in blob:
        raise ValueError('train_output must contain config and output dictionaries.')
    if 'model' not in blob['output'] or 'rules' not in blob['output'] or blob['output']['rules'] is None:
        raise ValueError('train_output must contain output.model and output.rules.')

    cfg = copy.deepcopy(blob['config'])
    rules = blob['output']['rules']

    trees_eval = sample_trees(num_data=int(eval_size), rules=rules, prior=None, probs=None, seed=int(eval_seed))

    data_cfg = copy.deepcopy(cfg)
    data_cfg.train_size = int(eval_size)
    data_cfg.test_size = 0
    data_cfg.batch_size = max(1, min(int(batch_size), int(eval_size)))
    loader, _ = init.init_data(trees_eval[cfg.num_layers], trees_eval[0], data_cfg)

    model = init.init_model(cfg)
    model.load_state_dict(blob['output']['model'])
    model = model.to(device_name).eval()
    for p in model.parameters():
        p.requires_grad = False

    return model, loader


def load_sae_set(ckpt_path, input_dim, device_name):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    if 'sae_state' not in ckpt or 'sae_layers' not in ckpt:
        raise ValueError(f'Invalid SAE checkpoint format: {ckpt_path}')

    sae_layers = [int(x) for x in ckpt['sae_layers']]
    sae_metrics = ckpt.get('sae_metrics', {})

    modules = {}
    for layer in sae_layers:
        state = ckpt['sae_state'].get(layer) or ckpt['sae_state'][str(layer)]
        metrics = sae_metrics.get(layer) or sae_metrics.get(str(layer)) or {}
        latent_dim = int(metrics.get('latent_dim') or state['encoder.weight'].shape[0])

        sae = models.SparseAutoencoder(input_dim=input_dim, latent_dim=latent_dim).to(device_name)
        sae.load_state_dict(state)
        sae.eval()
        for p in sae.parameters():
            p.requires_grad = False
        modules[layer] = sae

    return modules, sae_layers

In [20]:
def compute_activity_stats(model, loader, sae_modules, sae_layers, mode, device_name):
    """Compute dead-feature and mean-active-feature statistics over the eval loader.

    A feature is considered active when its value > 0, where the feature value is
    defined as  z_i * ||d_i||  (hidden activation times decoder column norm),
    matching the weighted L1 penalty used in the SAE loss.
    """
    assert mode in {'all_tokens', 'cls_token'}
    ever_active  = {l: torch.zeros(sae_modules[l].latent_dim, dtype=torch.bool, device=device_name) for l in sae_layers}
    active_sum   = {l: 0.0 for l in sae_layers}
    active_count = {l: 0   for l in sae_layers}

    # Pre-compute decoder column norms for each SAE layer (shape: latent_dim,)
    dec_norms = {l: sae_modules[l].decoder_feature_norms().to(device_name) for l in sae_layers}

    buffers = {l: [] for l in sae_layers}
    hooks = [model.blocks[l].register_forward_hook(lambda _m, _i, o, layer=l: buffers[layer].append(o.detach()))
             for l in sae_layers]

    with torch.no_grad():
        for x_batch, _ in loader:
            model(x_batch.to(device_name))
            for l in sae_layers:
                if not buffers[l]:
                    continue
                act = buffers[l].pop(0)
                act = act[:, :1, :] if mode == 'cls_token' else act[:, 1:, :]
                act = act.reshape(-1, act.size(-1))
                if act.numel() == 0:
                    continue
                _, z = sae_modules[l](act)
                # Feature value: z_i * ||d_i||  (matches the SAE weighted L1 loss)
                features  = z * dec_norms[l].unsqueeze(0)   # (N, latent_dim)
                is_active = features > 0
                active_sum[l]   += is_active.float().sum().item()
                active_count[l] += is_active.numel()
                ever_active[l]  |= is_active.any(dim=0)

    for h in hooks:
        h.remove()

    return {
        l: {
            'latent_dim':                int(sae_modules[l].latent_dim),
            'dead_features':             int((~ever_active[l]).sum()),
            'dead_feature_ratio':        float((~ever_active[l]).sum()) / max(sae_modules[l].latent_dim, 1),
            'mean_active_feature_ratio': active_sum[l] / max(active_count[l], 1),
        }
        for l in sae_layers
    }


def print_stats(title, stats):
    print('=' * 88)
    print(title)
    print('-' * 88)
    print(f"{'layer':>6} | {'latent_dim':>10} | {'dead_features':>13} | {'dead_ratio':>10} | {'mean_active_ratio':>17}")
    print('-' * 88)
    for l in sorted(stats):
        s = stats[l]
        print(f"{l:6d} | {s['latent_dim']:10d} | {s['dead_features']:13d} | {s['dead_feature_ratio']:10.6f} | {s['mean_active_feature_ratio']:17.6f}")

In [ ]:
for p in [train_output, sae_output_path_all_tokens, sae_output_path_cls_token]:
    if not Path(p).exists():
        raise FileNotFoundError(f'Missing required artifact: {p}')

model, eval_loader = load_transformer_and_loader(
    train_output_path=train_output,
    eval_size=analysis_eval_size,
    eval_seed=analysis_eval_seed_sample,
    batch_size=analysis_batch_size,
    device_name=analysis_device,
)

all_modules, all_layers = load_sae_set(sae_output_path, model.embedding_dim, analysis_device)

all_stats = compute_activity_stats(model, eval_loader, all_modules, all_layers, 'all_tokens', analysis_device)

print(f'Analysis eval size: {analysis_eval_size}, seed: {analysis_eval_seed_sample}, batch_size: {analysis_batch_size}')
print_stats('All-tokens SAE stats', all_stats)

/tmp/2663192/ipykernel_3620858/434515208.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = torch.load(train_output_path, map_location='cpu')


# parameters: 9465872


/tmp/2663192/ipykernel_3620858/434515208.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location='cpu')


Analysis eval size: 32768, seed: 98765432, batch_size: 256
All-tokens SAE stats
----------------------------------------------------------------------------------------
 layer | latent_dim | dead_features | dead_ratio | mean_active_ratio
----------------------------------------------------------------------------------------
     0 |      10240 |          8916 |   0.870703 |          0.000001
     1 |      10240 |          2391 |   0.233496 |          0.001252
     2 |      10240 |          5476 |   0.534766 |          0.068981
CLS-token SAE stats
----------------------------------------------------------------------------------------
 layer | latent_dim | dead_features | dead_ratio | mean_active_ratio
----------------------------------------------------------------------------------------
     0 |      10240 |         10231 |   0.999121 |          0.000000
     1 |      10240 |          8431 |   0.823340 |          0.000204
     2 |      10240 |          5625 |   0.549316 |          0

## SAE Impact on Classification Performance

For each SAE layer, we hook into the transformer and **replace the block output** with the SAE reconstruction (encoder → ReLU → decoder) for the token positions the SAE was trained on:
- `all_tokens` SAE: replaces positions 1..T (input tokens)
- `cls_token` SAE: replaces position 0 (CLS token)

Since classification reads from the CLS token at the end, both types of interventions propagate forward through subsequent blocks and affect the final logits.

**Normalized test error** = `err_with_sae / err_baseline`. A value of 1 means no degradation; larger values indicate reconstruction loss hurts classification.

In [ ]:
import torch.nn.functional as F


def eval_classification(model, loader, device_name, intervention_hooks=None):
    """Run inference and return (accuracy, mean cross-entropy loss).
    intervention_hooks: optional list of (module, hook_fn) applied during forward passes.
    """
    handles = []
    if intervention_hooks:
        for module, fn in intervention_hooks:
            handles.append(module.register_forward_hook(fn))

    correct = total = 0
    total_ce = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device_name), y.to(device_name)
            logits  = model(x)
            total_ce += F.cross_entropy(logits, y, reduction='sum').item()
            correct  += (logits.argmax(-1) == y).sum().item()
            total    += y.size(0)

    for h in handles:
        h.remove()

    return correct / total, total_ce / total


def make_sae_hook(sae, mode):
    """Return a forward hook that replaces block output with SAE reconstruction
    for the token positions the SAE was trained on."""
    def hook(_m, _i, output):
        out = output.clone()
        if mode == 'cls_token':
            flat = out[:, :1, :].reshape(-1, out.size(-1))
            recon, _ = sae(flat)
            out[:, :1, :] = recon.reshape(output[:, :1, :].shape)
        else:  # all_tokens: positions 1..T
            flat = out[:, 1:, :].reshape(-1, out.size(-1))
            recon, _ = sae(flat)
            out[:, 1:, :] = recon.reshape(output[:, 1:, :].shape)
        return out
    return hook


# ── Random-chance baseline ────────────────────────────────────────────────────
# A random classifier has error 1 - 1/v, where v = number of classes.
random_err = 1.0 - 1.0 / model.num_classes
print(f'Number of classes v={model.num_classes},  random error = {random_err:.6f}')

# ── Baseline (no SAE intervention) ───────────────────────────────────────────
baseline_acc, baseline_ce = eval_classification(model, eval_loader, analysis_device)
baseline_err = 1.0 - baseline_acc
baseline_std = baseline_err / random_err
print(f'Baseline  acc={baseline_acc:.4f}  err={baseline_err:.6f}  std_err={baseline_std:.4f}  CE={baseline_ce:.4f}\n')

# ── Per-layer and all-layers-stacked interventions ────────────────────────────
header = f"{'layer':>6} | {'acc':>8} | {'err':>10} | {'std_err':>9} | {'CE':>8}"
sep    = '-' * len(header)

for label, sae_modules, sae_layers_list, mode in [
    ('all_tokens', all_modules, all_layers, 'all_tokens')
]:
    print(f'== {label} SAE ==')
    print(header)
    print(sep)

    for layer in sae_layers_list:
        hook_fn = make_sae_hook(sae_modules[layer], mode)
        acc, ce = eval_classification(
            model, eval_loader, analysis_device,
            intervention_hooks=[(model.blocks[layer], hook_fn)],
        )
        err     = 1.0 - acc
        std_err = err / random_err
        print(f"{layer:>6} | {acc:>8.4f} | {err:>10.6f} | {std_err:>9.4f} | {ce:>8.4f}")

    # All layers stacked simultaneously
    all_hooks = [(model.blocks[l], make_sae_hook(sae_modules[l], mode)) for l in sae_layers_list]
    acc, ce = eval_classification(model, eval_loader, analysis_device, intervention_hooks=all_hooks)
    err     = 1.0 - acc
    std_err = err / random_err
    print(sep)
    print(f"{'all':>6} | {acc:>8.4f} | {err:>10.6f} | {std_err:>9.4f} | {ce:>8.4f}  ← all layers stacked")
    print()

Number of classes v=16,  random error = 0.937500
Baseline  acc=1.0000  err=0.000000  std_err=0.0000  CE=0.0000

== all_tokens SAE ==
 layer |      acc |        err |   std_err |       CE
-----------------------------------------------------
     0 |   0.0606 |   0.939362 |    1.0020 |   6.8282
     1 |   0.9999 |   0.000061 |    0.0001 |   0.0003
     2 |   1.0000 |   0.000000 |    0.0000 |   0.0000
-----------------------------------------------------
   all |   0.0673 |   0.932739 |    0.9949 |  11.1816  ← all layers stacked

== cls_token SAE ==
 layer |      acc |        err |   std_err |       CE
-----------------------------------------------------
     0 |   1.0000 |   0.000000 |    0.0000 |   0.0000
     1 |   1.0000 |   0.000000 |    0.0000 |   0.0000
     2 |   1.0000 |   0.000000 |    0.0000 |   0.0000
-----------------------------------------------------
   all |   0.9999 |   0.000092 |    0.0001 |   0.0002  ← all layers stacked

